In [1]:
%pip install -q datasets tiktoken wandb huggingface_hub

Note: you may need to restart the kernel to use updated packages.


In [2]:
import math
import time
from pathlib import Path
import sys

import torch
import torch.nn.functional as F
import tiktoken
PROJECT_ROOT = next((path for path in (Path.cwd(), *Path.cwd().parents)
                     if (path / 'src' / 'llm_mini_lab').is_dir()), None)
if PROJECT_ROOT is None:
    raise FileNotFoundError('Could not find the llm-mini-lab project root')
sys.path.insert(0, str(PROJECT_ROOT / 'src'))

from llm_mini_lab.pretraining import (
    create_dataloader_fineweb, make_fixed_eval_loaders, GPT_CONFIG_124M)
from llm_mini_lab.model import GPTModel

device = torch.device("cuda" if torch.cuda.is_available()
                      else "mps" if torch.backends.mps.is_available()
                      else "cpu")
print("PyTorch:", torch.__version__)
print("Device:", device)
if device.type != "cuda":
    raise RuntimeError("This optimized notebook requires an NVIDIA CUDA GPU.")

torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.benchmark = True
print("GPU:", torch.cuda.get_device_name(0))
print("VRAM (GiB):", round(torch.cuda.get_device_properties(0).total_memory / 2**30, 1))

PyTorch: 2.11.0+cu128
Device: cuda
GPU: NVIDIA TITAN RTX
VRAM (GiB): 23.5


In [5]:
# ── Configuration (adjust to taste) ──
MAX_DOCS    = None    # None trains on the complete streamed FineWeb-Edu dataset.
MAX_LENGTH  = 512    # Good throughput/context trade-off for a 24-GB Titan RTX.
MICRO_BATCH_SIZE = 8 # If CUDA OOMs, reduce this to 4.
GRAD_ACCUM_STEPS = 2 # Effective batch: 16 sequences = 8,192 tokens/update.
VAL_MOD     = 100     # 1 validation document every 100
SEED        = 123
NUM_WORKERS = 0      # Parallel CPU tokenization and streaming.
EVAL_FREQ   = 250    # Optimizer updates between evaluations.
LOG_EVERY = 10       # Fine-grained training-loss logging (optimizer updates).
GLOBAL_LOG_SECONDS = 20  # Rolling global metrics interval, in wall-clock seconds.
EVAL_BATCHES = 8
SAVE_EVERY = 1_000   # Local safety checkpoint interval; the same latest file is overwritten.
HF_SAVE_EVERY = 20_000  # Remote recovery snapshots; each snapshot is kept on Hugging Face.
SCHEDULE_STEPS = 122_000  # Approx. one 1B-token epoch at this effective batch.
WARMUP_STEPS = 2_000
LEARNING_RATE = 1.5e-4  # Conservative for FP16 training with this custom model.
WEIGHT_DECAY = 0.1
# Optional: paste the existing W&B run ID to continue its same dashboard.
WANDB_RUN_ID = None
# ────────────────────────────────────

train_loader, val_loader = create_dataloader_fineweb(
    batch_size=MICRO_BATCH_SIZE, max_length=MAX_LENGTH, val_mod=VAL_MOD,
    seed=SEED, max_docs=MAX_DOCS, num_workers=NUM_WORKERS)

train_eval_loader, val_eval_loader = make_fixed_eval_loaders(
    train_loader, val_loader, max_train_batches=8, max_val_batches=16)

xb, yb = next(iter(train_loader))
print("Micro-batch:", xb.shape, "| target == shifted input:",
      torch.equal(yb[:, :-1], xb[:, 1:]))
print(f"Effective tokens/update: {MICRO_BATCH_SIZE * MAX_LENGTH * GRAD_ACCUM_STEPS:,}")

Micro-batch: torch.Size([8, 512]) | target == shifted input: True
Effective tokens/update: 8,192


In [6]:
from datasets import load_dataset

enc = tiktoken.get_encoding("gpt2")
stream = load_dataset("codelion/fineweb-edu-1B", split="train", streaming=True)
stream = stream.shuffle(seed=SEED, buffer_size=10000)

n_docs, n_tokens = 0, 0
for ex in stream:
    n_tokens += len(enc.encode(ex["text"]))
    n_docs += 1
    if n_docs >= 100:
        break

avg = n_tokens / n_docs
print(f"Average: {avg:.0f} tokens/document (sample of {n_docs} docs)")
if MAX_DOCS is None:
    print("Training on the complete streamed dataset (no document limit).")
else:
    est_tokens = avg * MAX_DOCS
    print(f"Estimated training tokens (max_docs={MAX_DOCS}): ~{est_tokens/1e6:.1f}M")
    print(f"Estimated microbatches (batch {MICRO_BATCH_SIZE} x {MAX_LENGTH}): ~{est_tokens / (MICRO_BATCH_SIZE * MAX_LENGTH):.0f}")

# Tracking and publishing for an external GPU. Credentials are requested
# interactively and are never stored in this notebook.
import wandb
from huggingface_hub import HfApi, create_repo, login

wandb.login()
login()

hf_api = HfApi()
hf_username = hf_api.whoami()["name"]
hf_repo_id = f"{hf_username}/gpt2-fineweb-124m"
create_repo(hf_repo_id, private=True, exist_ok=True)

run_config = {
    "model": "GPT-2 124M",
    "dataset": "codelion/fineweb-edu-1B",
    "max_docs": MAX_DOCS,
    "context_length": MAX_LENGTH,
    "micro_batch_size": MICRO_BATCH_SIZE,
    "gradient_accumulation_steps": GRAD_ACCUM_STEPS,
    "effective_batch_size": MICRO_BATCH_SIZE * GRAD_ACCUM_STEPS,
    "learning_rate": LEARNING_RATE,
    "weight_decay": WEIGHT_DECAY,
}
wandb_init_kwargs = {"project": "gpt2-fineweb-pretraining", "config": run_config}
if WANDB_RUN_ID:
    wandb_init_kwargs.update(id=WANDB_RUN_ID, resume="must")
    print(f"Resuming W&B run: {WANDB_RUN_ID}")
wandb.init(**wandb_init_kwargs)
wandb.define_metric("update")
wandb.define_metric("*", step_metric="update")

Average: 1076 tokens/document (sample of 100 docs)
Training on the complete streamed dataset (no document limit).


wandb: ERROR Failed to detect the name of this notebook. You can set it manually with the WANDB_NOTEBOOK_NAME environment variable to enable code saving.
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /root/.netrc.
wandb: Currently logged in as: volverman22 (volverman22-universidad-aut-noma-de-madrid) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


In [7]:
cfg = {**GPT_CONFIG_124M, "context_length": MAX_LENGTH}

torch.manual_seed(SEED)
model = GPTModel(cfg).to(device)
# GPT-2 shares token embedding and output-head weights: ~124M parameters, not ~162M.
model.out_head.weight = model.tok_emb.weight

n_params = sum(p.numel() for p in model.parameters())
print(f"Model parameters: {n_params:,}  ({n_params/1e6:.1f}M)")

optimizer = torch.optim.AdamW(
    model.parameters(), lr=LEARNING_RATE, betas=(0.9, 0.95), weight_decay=WEIGHT_DECAY
)

def lr_multiplier(step):
    if step < WARMUP_STEPS:
        return (step + 1) / WARMUP_STEPS
    progress = (step - WARMUP_STEPS) / (SCHEDULE_STEPS - WARMUP_STEPS)
    return 0.1 + 0.9 * 0.5 * (1 + math.cos(math.pi * min(progress, 1.0)))

scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda=lr_multiplier)
scaler = torch.amp.GradScaler("cuda")
tokenizer = tiktoken.get_encoding("gpt2")

Model parameters: 124,018,944  (124.0M)


In [ ]:
def evaluate(loader, max_batches):
    model.eval()
    losses = []
    with torch.inference_mode():
        for batch_idx, (x, y) in enumerate(loader):
            if batch_idx >= max_batches:
                break
            x, y = x.to(device, non_blocking=True), y.to(device, non_blocking=True)
            with torch.autocast(device_type="cuda", dtype=torch.float16):
                loss = F.cross_entropy(model(x).flatten(0, 1), y.flatten())
            losses.append(loss.float().item())
    model.train()
    return sum(losses) / len(losses)

checkpoint_dir = PROJECT_ROOT / "checkpoints"
checkpoint_dir.mkdir(exist_ok=True)

def save_checkpoint(update_step, microbatch_idx, tokens_seen, history, suffix):
    path = checkpoint_dir / f"gpt2-fineweb-124m-{suffix}.pt"
    torch.save({
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "scheduler_state_dict": scheduler.state_dict(),
        "scaler_state_dict": scaler.state_dict(),
        "update_step": update_step,
        "microbatch_idx": microbatch_idx,
        "tokens_seen": tokens_seen,
        "history": history,
        "config": {**cfg, **run_config},
        "wandb_run_id": wandb.run.id if wandb.run else None,
    }, path)
    return path

history = {"step": [], "tokens": [], "train_loss": [], "val_loss": []}
tokens_seen = 0
running_loss = 0.0
loss_in_update = 0.0
global_loss_sum = 0.0
global_update_count = 0
last_global_log_time = time.perf_counter()
tokens_at_last_global_log = 0
update_step = 0



In [ ]:
# ── Restore from checkpoint (optional; run this cell before training) ──
# Set to None to start a new training run.
RESUME_FROM = PROJECT_ROOT / "checkpoints" / "gpt2-fineweb-124m-latest.pt"

resume_microbatch_idx = 0
if RESUME_FROM is not None:
    resume_path = Path(RESUME_FROM)
    if not resume_path.is_file():
        raise FileNotFoundError(f"Checkpoint not found: {resume_path.resolve()}")
    try:
        checkpoint = torch.load(resume_path, map_location=device, weights_only=False)
    except TypeError:  # Compatibility with older PyTorch versions.
        checkpoint = torch.load(resume_path, map_location=device)

    saved_config = checkpoint.get("config", {})
    expected_config = {
        "context_length": MAX_LENGTH,
        "micro_batch_size": MICRO_BATCH_SIZE,
        "gradient_accumulation_steps": GRAD_ACCUM_STEPS,
    }
    for key, current_value in expected_config.items():
        saved_value = saved_config.get(key)
        if saved_value is not None and saved_value != current_value:
            raise ValueError(f"Checkpoint {key}={saved_value}, but notebook uses {current_value}.")

    model.load_state_dict(checkpoint["model_state_dict"])
    optimizer.load_state_dict(checkpoint["optimizer_state_dict"])
    scheduler.load_state_dict(checkpoint["scheduler_state_dict"])
    scaler.load_state_dict(checkpoint["scaler_state_dict"])
    update_step = int(checkpoint["update_step"])
    tokens_seen = int(checkpoint["tokens_seen"])
    history = checkpoint.get("history", history)
    # Older checkpoints do not contain this field; full updates use this exact fallback.
    resume_microbatch_idx = int(
        checkpoint.get("microbatch_idx", update_step * GRAD_ACCUM_STEPS))
    if resume_microbatch_idx != update_step * GRAD_ACCUM_STEPS:
        raise ValueError("Only checkpoints saved after a complete optimizer update can be resumed.")
    tokens_at_last_global_log = tokens_seen
    print(f"Resuming from {resume_path}: update {update_step:,}, "
          f"{tokens_seen / 1e6:.2f}M tokens. Replaying and skipping "
          f"{resume_microbatch_idx:,} microbatches to restore the stream position.")

model.train()
optimizer.zero_grad(set_to_none=True)


In [ ]:
for microbatch_idx, (x, y) in enumerate(train_loader, start=1):
    # The stream has no serializable cursor; replay it deterministically and
    # discard batches consumed before the checkpoint.
    if microbatch_idx <= resume_microbatch_idx:
        if microbatch_idx % 1_000 == 0:
            print(f"Replaying stream: skipped {microbatch_idx:,}/{resume_microbatch_idx:,} microbatches")
        continue
    x, y = x.to(device, non_blocking=True), y.to(device, non_blocking=True)
    with torch.autocast(device_type="cuda", dtype=torch.float16):
        logits = model(x)
        loss = F.cross_entropy(logits.flatten(0, 1), y.flatten())
        scaled_loss = loss / GRAD_ACCUM_STEPS
    if not torch.isfinite(loss):
        raise FloatingPointError(
            f"Non-finite loss before optimizer update at microbatch {microbatch_idx}; "
            "stop and resume from the latest finite checkpoint.")
    scaler.scale(scaled_loss).backward()
    running_loss += loss.detach().float().item()
    loss_in_update += loss.detach().float().item()
    tokens_seen += x.numel()

    if microbatch_idx % GRAD_ACCUM_STEPS != 0:
        continue

    scaler.unscale_(optimizer)
    grad_norm = torch.nn.utils.clip_grad_norm_(
        model.parameters(), max_norm=1.0, error_if_nonfinite=False)
    if not torch.isfinite(grad_norm):
        print(f"Non-finite gradients at update {update_step + 1:,}; skipping update and reducing AMP scale.")
        optimizer.zero_grad(set_to_none=True)
        scaler.update()
        running_loss -= loss_in_update
        loss_in_update = 0.0
        continue
    scaler.step(optimizer)
    scaler.update()
    optimizer.zero_grad(set_to_none=True)
    scheduler.step()
    update_step += 1
    update_loss = loss_in_update / GRAD_ACCUM_STEPS
    loss_in_update = 0.0
    global_loss_sum += update_loss
    global_update_count += 1

    if update_step % LOG_EVERY == 0:
        wandb.log({"update": update_step, "train/minibatch_loss": update_loss,
                   "learning_rate": scheduler.get_last_lr()[0]})

    now = time.perf_counter()
    elapsed_global = now - last_global_log_time
    if elapsed_global >= GLOBAL_LOG_SECONDS:
        tokens_per_second = (tokens_seen - tokens_at_last_global_log) / elapsed_global
        wandb.log({"update": update_step,
                   "train/global_loss": global_loss_sum / global_update_count,
                   "throughput/tokens_per_second": tokens_per_second,
                   "throughput/updates_per_second": global_update_count / elapsed_global})
        print(f"Update {update_step:,} | global train {global_loss_sum / global_update_count:.3f} | "
              f"{tokens_per_second:,.0f} tokens/s")
        global_loss_sum = 0.0
        global_update_count = 0
        last_global_log_time = now
        tokens_at_last_global_log = tokens_seen

    if update_step % EVAL_FREQ == 0:
        train_loss = running_loss / (EVAL_FREQ * GRAD_ACCUM_STEPS)
        val_loss = evaluate(val_eval_loader, EVAL_BATCHES)
        history["step"].append(update_step)
        history["tokens"].append(tokens_seen)
        history["train_loss"].append(train_loss)
        history["val_loss"].append(val_loss)
        wandb.log({"update": update_step, "tokens_seen": tokens_seen,
                   "train/loss": train_loss, "validation/loss": val_loss,
                   "learning_rate": scheduler.get_last_lr()[0],
                   "gradient_norm": grad_norm.item()})
        print(f"Update {update_step:,} | tokens {tokens_seen / 1e6:.2f}M | "
              f"train {train_loss:.3f} | val {val_loss:.3f}")
        running_loss = 0.0

    if update_step % SAVE_EVERY == 0:
        local_checkpoint = save_checkpoint(update_step, microbatch_idx, tokens_seen, history, "latest")
        print(f"Local checkpoint saved: {local_checkpoint}")

    if update_step % HF_SAVE_EVERY == 0:
        hf_api.upload_file(path_or_fileobj=str(local_checkpoint),
                           path_in_repo=f"gpt2-fineweb-124m-step-{update_step}.pt", repo_id=hf_repo_id,
                           commit_message=f"Checkpoint at update {update_step:,}")
        print(f"Checkpoint uploaded to Hugging Face: gpt2-fineweb-124m-step-{update_step}.pt")

checkpoint_path = save_checkpoint(update_step, microbatch_idx, tokens_seen, history, "latest")
hf_api.upload_file(path_or_fileobj=str(checkpoint_path), path_in_repo="gpt2-fineweb-124m-final.pt",
                   repo_id=hf_repo_id, commit_message="Upload final GPT-2 FineWeb checkpoint")
wandb.save(str(checkpoint_path))
wandb.finish()

train_losses, val_losses, tokens_seen = history["train_loss"], history["val_loss"], history["tokens"]
print(f"Training complete: {update_step:,} optimizer updates; final checkpoint: {checkpoint_path}")

In [ ]:
print("=" * 55)
print(f"Training tokens seen: {tokens_seen[-1]:,}  ({tokens_seen[-1]/1e6:.1f}M)")
print(f"Final train loss: {train_losses[-1]:.3f}")
print(f"Final val loss:   {val_losses[-1]:.3f}")
print("=" * 55)
print(f"✅ Final checkpoint: {checkpoint_path}")
print(f"✅ Hugging Face repository: {hf_repo_id}")

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(8, 4))
plt.plot(tokens_seen, train_losses, label="Train loss")
plt.plot(tokens_seen, val_losses, linestyle="-.", label="Val loss")
plt.xlabel("Tokens seen")
plt.ylabel("Loss")
plt.legend()
plt.grid(alpha=0.3)
plt.show()